# OjaLM-1 Fine-Tuning (AfriqueQwen)
Run these cells in order. The first cell will prompt you to upload `training_corpus.jsonl`.

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to c:\users\olusegun\appdata\local\temp\pip-install-2rq0cm78\unsloth_67e07007a3f745c49e73a56f51d96365
  Resolved https://github.com/unslothai/unsloth.git to commit 1c77b4d1496fe5d7f26bf624315acc682ef5cd6e
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached typer-0.27.0-py3-none-any.whl.metadata (15 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached tyro-1.0.15-py3-none-any.whl.metadata (12 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached datase

  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git 'C:\Users\olusegun\AppData\Local\Temp\pip-install-2rq0cm78\unsloth_67e07007a3f745c49e73a56f51d96365'
  error: subprocess-exited-with-error
  
  × Building wheel for unsloth (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [7793 lines of output]
      toml section missing WindowsPath('pyproject.toml') does not contain a tool.setuptools_scm section
      running bdist_wheel
      running build
      running build_py
      creating build\lib\studio
      copying studio\install_llama_prebuilt.py -> build\lib\studio
      copying studio\install_node_prebuilt.py -> build\lib\studio
      copying studio\install_python_stack.py -> build\lib\studio
      copying studio\__init__.py -> build\lib\studio
      creating build\lib\unsloth
      copying unsloth\chat_templates.py -> build\lib\unsloth
      copying unsloth\device_type.py -> build\lib\unsloth
      copying unsloth\import_fixes

  Using cached trl-1.8.0-py3-none-any.whl.metadata (12 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl.metadata (10 kB)
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 MB 119.8 kB/s eta 0:00:22
   ---------------------------------------- 0.0/2.6 MB 119.8 kB/s eta 0:00:22
    --------------------------------------- 0.0/2.6 MB 131.3 kB/s eta 0:00:20
    --------------------------------------- 0.1/2.6 MB 182.2 kB/s eta 0:00:15
   - -------------------------------------- 0.1/2.6 MB 298.5 kB/s eta 0:00:09
   


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from google.colab import files
print('Please upload your training_corpus.jsonl file here:')
uploaded = files.upload()

ModuleNotFoundError: No module named 'google'

In [ ]:
import os
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

tokenizer = get_chat_template(
    tokenizer, 
    chat_template='chatml', 
    mapping={'role': 'from', 'content': 'value', 'user': 'human', 'assistant': 'gpt'}
)

def formatting_prompts_func(examples):
    convos = examples['conversations']
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]
    return { 'text' : texts }

dataset = load_dataset('json', data_files='training_corpus.jsonl', split='train')
dataset = dataset.map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    tokenizer = tokenizer,
    dataset_text_field = 'text',
    max_seq_length = 2048,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 60,
        learning_rate = 2e-4,
        output_dir = 'outputs',
        optim = 'adamw_8bit',
        seed = 3407,
    ),
)
trainer.train()

model.save_pretrained('ojalm-v1-lora')
tokenizer.save_pretrained('ojalm-v1-lora')
print('Training Complete!')
